# Paper §5.3 — Routing-and-integration synthesis

Narrative-only notebook that loads the §5.1 + §5.2 outputs and lays out the routing-and-integration framework the paper uses to synthesize them. No new compute.

**Spec source-of-truth.** `docs/paper/emnlp_outline_ko.md` — §5.3 (Routing-and-integration).

This notebook drives the heavy inference stages by `subprocess`-invoking
the existing sharded drivers in `scripts/`. The `RUN_INFERENCE = False`
toggle below lets a reviewer read the entire pipeline without GPU
access. Full reproduction targets the **8 × H200** cluster and uses
`--gpus 0,1,2,3,4,5,6,7` end-to-end.


## 1 · Setup — paths, GPU sharding, subprocess helper

In [1]:
from __future__ import annotations
import json
import os
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_main_worktree() -> Path:
    # Gitignored artifacts (inputs/, outputs/, docs/insights/_data/) live in
    # the main worktree even when this notebook runs from a linked worktree.
    common = subprocess.check_output(
        ["git", "rev-parse", "--git-common-dir"], cwd=Path.cwd(), text=True
    ).strip()
    return Path(common).resolve().parent


def find_worktree_root() -> Path:
    # Current worktree's working tree top (== main when not in a linked worktree).
    return Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=Path.cwd(), text=True
    ).strip()).resolve()


MAIN     = find_main_worktree()
WORKTREE = find_worktree_root()

# Outputs live under MAIN (gitignored); figures land under WORKTREE so they ride the active branch.
SCRIPTS    = MAIN / "scripts"
CONFIGS    = MAIN / "configs"
DATA_DIR   = MAIN / "docs" / "insights" / "_data"
PRED_ROOT  = MAIN / "outputs" / "paper" / "cross_model_cross_dataset" / "predictions"

# Legacy pooled trees — §5.2 subspace sweep still writes here. §5.2
# isolation is tracked as a follow-up (the sharded drivers and the two
# `aggregate_e6_*.py` scripts also hardcode this root).
E6_ROOT = MAIN / "outputs" / "e6_steering"

# Reproducer-isolated output roots — the legacy `outputs/attention_analysis/`
# tree pools many old runs that the §5.1 aggregators would otherwise mix in.
# §5.1 stages all point at this fresh root so the §5.1 notebook's outputs
# only reflect *this* re-run.
ATT_ROOT_FRESH = MAIN / "outputs" / "paper" / "section_5_attention"
PEAKS_CSV      = MAIN / "outputs" / "paper" / "section_5_attention" / "_data" / "cross_dataset_peaks.csv"

ATT_ROOT_FRESH.mkdir(parents=True, exist_ok=True)
PEAKS_CSV.parent.mkdir(parents=True, exist_ok=True)

PDF_OUT = MAIN     / "outputs" / "paper" / "section_5_figures"
PNG_OUT = WORKTREE / "docs"    / "figures"
PDF_OUT.mkdir(parents=True, exist_ok=True)
PNG_OUT.mkdir(parents=True, exist_ok=True)

GPUS = os.environ.get("VLM_ANCHOR_GPUS", "0,1,2,3,4,5,6,7")  # 8 GPUs by default
RUN_INFERENCE = False  # set True to invoke the heavy sharded drivers.

print(f"MAIN     = {MAIN}")
print(f"WORKTREE = {WORKTREE}")
print(f"GPUS     = {GPUS}")
print(f"RUN_INFERENCE = {RUN_INFERENCE}")


MAIN     = /mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing
WORKTREE = /mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing/.claude/worktrees/paper-section-5-reproduction
GPUS     = 0,1,2,3,4,5,6,7
RUN_INFERENCE = False


In [2]:
def run_cmd(cmd: list[str] | str, *, dry: bool = False, env: dict | None = None) -> int:
    # Print and (optionally) execute a shell command from the main worktree.
    printable = " ".join(shlex.quote(c) for c in cmd) if isinstance(cmd, list) else cmd
    print(f"$ {printable}")
    if dry:
        print("  (dry — RUN_INFERENCE=False)")
        return 0
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    return subprocess.run(cmd, cwd=MAIN, env=full_env,
                          shell=isinstance(cmd, str)).returncode


def save_figure(fig, stem: str):
    pdf = PDF_OUT / f"{stem}.pdf"
    png = PNG_OUT / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, bbox_inches="tight", dpi=160)
    print(f"wrote {pdf}")
    print(f"wrote {png}")


## 2 · Recap — what §5.1 + §5.2 established

**§5.1 (per-model peak heterogeneity).** Across the 5-model mechanism
panel the attention peak layer varies from early (Gemma3-4b ≈ L=5 / 42)
to mid (ConvLLaVA, LLaVA-1.5, FastVLM around mid-stack) to late
(OneVision ≈ L=27 / 28 on PlotQA + TallyQA). OneVision additionally
shows a *dataset-dependent* peak shift: InfoVQA pushes the peak back
to L≈14. No uniform causal site.

**§5.2 (within-layer multi-direction).** The K-subspace sweep on
OneVision Main shows a monotonic improvement K=1 → K=2 → K=4 → K=8 in
Δadopt(a) and Δdf(a) at the chosen integration site (L=26, α=1.0). At
L=26 the K=1 fallback fails to clear the anchoring effect across 5
datasets — single direction is insufficient (Figure §5.2b).


In [3]:
peaks_path = PEAKS_CSV  # isolated reproducer artifact, falls back to legacy below
if not peaks_path.exists():
    legacy = DATA_DIR / "cross_dataset_peaks.csv"
    if legacy.exists():
        print(f"  (fresh CSV missing; reading legacy {legacy} for smoke-only preview)")
        peaks_path = legacy
sweep_path = DATA_DIR / "p4_layer_sweep_per_cell_ci.csv"
pilot_path = DATA_DIR / "e6_pilot_grid_plotqa.csv"

if peaks_path.exists():
    peaks = pd.read_csv(peaks_path)
    print(f"§5.1 peaks rows: {len(peaks)} (source: {peaks_path})")
else:
    print(f"missing: {peaks_path} — run paper_section_5_1_attention_peaks.ipynb first.")
if pilot_path.exists():
    pilot = pd.read_csv(pilot_path)
    print(f"§5.2a pilot cells: {len(pilot)}")
else:
    print(f"missing: {pilot_path} — run paper_section_5_2_subspace_sweep.ipynb first.")
if sweep_path.exists():
    sweep = pd.read_csv(sweep_path)
    print(f"§5.2b sweep cells: {len(sweep)}")
else:
    print(f"missing: {sweep_path} — run paper_section_5_2_subspace_sweep.ipynb first.")


  (fresh CSV missing; reading legacy /mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing/docs/insights/_data/cross_dataset_peaks.csv for smoke-only preview)
§5.1 peaks rows: 48 (source: /mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing/docs/insights/_data/cross_dataset_peaks.csv)
missing: /mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing/docs/insights/_data/e6_pilot_grid_plotqa.csv — run paper_section_5_2_subspace_sweep.ipynb first.
§5.2b sweep cells: 35


## 3 · Routing-and-integration framework

The two §5 findings co-jointly characterize the anchoring representation
as having two structural properties:

- **Routed across multiple attention layers** — model-specific peak
  layers indicate that different architectures handle anchoring at
  different stages of their forward pass. Uniformity would be the
  exception, not the rule, under this account; the §5.1 panel observes
  exactly that heterogeneity.
- **Integrated into a residual-stream subspace of dimension ≥ 2** —
  the K-monotonic improvement in §5.2 and the K=1 fallback failure
  rule out a single-direction representation. The integration site for
  OneVision Main is `L=26` of 28 layers (≈ 93 % depth), i.e., late in
  the residual stream.

**Mitigation design implications (§6).** The two structural properties
map directly onto the two §6 design choices:
- Choose the *integration site* per architecture (no shared causal site).
- Project out a *multi-direction subspace* (K ≥ 2), not a single
  direction. K=8 captures > 95 % of the explained variance on
  OneVision Main and is the chosen K in the paper's §6.2 mitigation.

**Limitations of §5 evidence.** The peak panel covers 5 architectures
on 3 datasets; further architectures (e.g., Gemma3-27b, Qwen2.5-VL-32b)
would extend the heterogeneity claim. The K-subspace sweep is run
on OneVision Main only; the K-monotonic property is *expected* to be
architecture-general but is *verified* on one model. Cross-architecture
directional verification (γ-β residual-stream bridge on Qwen3-VL) is in
Appendix E and confirms the *direction* of the mitigation but not its
magnitude.


## Summary

§5.3 is interpretation only; it depends on §5.1 + §5.2 outputs being
present on disk. The two §5 findings (peak heterogeneity + within-layer
multi-direction) form the routing-and-integration framework used by §6.

Re-run order:
1. `notebooks/paper_section_5_1_attention_peaks.ipynb` — produces `cross_dataset_peaks.csv`.
2. `notebooks/paper_section_5_2_subspace_sweep.ipynb` — produces `e6_pilot_grid_plotqa.csv` + `p4_layer_sweep_per_cell_ci.csv`.
3. This notebook — reads them back, restates the framework.
